# Problem 2 — Linear Regression in Amazon SageMaker

This notebook demonstrates two approaches:

1. Without a user-created container: train directly in the SageMaker notebook kernel and save/upload the artifact.
2. With container technology: build a custom Docker training image, push it to Amazon ECR, and start a SageMaker training job.

In [10]:
import os
import boto3
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Load the red-wine dataset from the shared folder
data = pd.read_csv("../winequality-red.csv", sep=";")

# Separate the 11 input features from the quality target
X = data.drop(columns=["quality"])
y = data["quality"]

# Create training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Dataset loaded successfully.")
print("Dataset shape:", data.shape)
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

data.head()

Dataset loaded successfully.
Dataset shape: (1599, 12)
Training records: 1279
Testing records: 320


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Part A — SageMaker notebook workflow without a user-created container

In [11]:
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ]
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

metrics_a = {
    "MAE": mean_absolute_error(y_test, predictions),
    "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
    "R2": r2_score(y_test, predictions),
}

print(f"MAE: {metrics_a['MAE']:.4f}")
print(f"RMSE: {metrics_a['RMSE']:.4f}")
print(f"R²: {metrics_a['R2']:.4f}")

MAE: 0.5035
RMSE: 0.6245
R²: 0.4032


In [12]:
joblib.dump(model, "linear_regression_no_custom_container.pkl")
print("Model artifact saved.")

Model artifact saved.


In [14]:
from pathlib import Path

import boto3
from sagemaker.core.helper.session_helper import Session

# Create the SageMaker SDK v3 session
session = Session()

bucket = session.default_bucket()
prefix = "wine-quality-linear-regression"

# The CSV is one folder above the notebook
data_path = Path("../winequality-red.csv")

# The saved model is in the notebook's current folder
model_path = Path("linear_regression_no_custom_container.pkl")

data_s3_uri = session.upload_data(
    path=str(data_path),
    bucket=bucket,
    key_prefix=f"{prefix}/input",
)

model_s3_uri = session.upload_data(
    path=str(model_path),
    bucket=bucket,
    key_prefix=f"{prefix}/artifacts",
)

print("Dataset S3 URI:", data_s3_uri)
print("Model S3 URI:", model_s3_uri)

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Dataset S3 URI: s3://amazon-sagemaker-275060988924-us-east-1-5o9dz22x4yjo6f/wine-quality-linear-regression/input/winequality-red.csv
Model S3 URI: s3://amazon-sagemaker-275060988924-us-east-1-5o9dz22x4yjo6f/wine-quality-linear-regression/artifacts/linear_regression_no_custom_container.pkl


### Interpretation

This part does not build or manage a custom Docker image. Training occurs in the notebook environment with Scikit-learn. The saved artifact can be placed in S3 for deployment or grading evidence.

## Part B — Custom container in SageMaker

In [15]:
import boto3

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)

# Create the SageMaker v3 session
sagemaker_session = Session()

region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
role = get_execution_role()

ecr_repository = "wine-quality-sagemaker"

image_uri = (
    f"{account_id}.dkr.ecr.{region}.amazonaws.com/"
    f"{ecr_repository}:latest"
)

print("Region:", region)
print("Account ID:", account_id)
print("Role:", role)
print("ECR repository:", ecr_repository)
print("Image URI:", image_uri)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Region: us-east-1
Account ID: 275060988924
Role: arn:aws:iam::275060988924:role/service-role/AmazonSageMakerAdminIAMExecutionRole
ECR repository: wine-quality-sagemaker
Image URI: 275060988924.dkr.ecr.us-east-1.amazonaws.com/wine-quality-sagemaker:latest


In [18]:
%%writefile train_sagemaker.py

import argparse
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--model-dir",
        type=str,
        default=os.environ.get("SM_MODEL_DIR"),
    )

    parser.add_argument(
        "--train",
        type=str,
        default=os.environ.get("SM_CHANNEL_TRAIN"),
    )

    args = parser.parse_args()

    train_path = Path(args.train)
    csv_files = list(train_path.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV training file found in {train_path}"
        )

    data = pd.read_csv(csv_files[0], sep=";")

    X = data.drop(columns=["quality"])
    y = data["quality"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
    )

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("regressor", LinearRegression()),
        ]
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    metrics = {
        "MAE": float(mean_absolute_error(y_test, predictions)),
        "RMSE": float(
            np.sqrt(mean_squared_error(y_test, predictions))
        ),
        "R2": float(r2_score(y_test, predictions)),
    }

    print("Training metrics:")
    print(json.dumps(metrics, indent=2))

    model_dir = Path(args.model_dir)
    model_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(model, model_dir / "model.joblib")

    with open(model_dir / "metrics.json", "w") as file:
        json.dump(metrics, file, indent=2)

    print("Model saved successfully.")


if __name__ == "__main__":
    main()

Writing train_sagemaker.py


### Build and push the custom image

Run the displayed shell commands from an environment with Docker, AWS CLI access, and permission to push to ECR.

In [21]:
import boto3

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)

# Create SageMaker session
sagemaker_session = Session()

# Define region
region = boto3.Session().region_name

# Define account
account_id = boto3.client("sts").get_caller_identity()["Account"]

# Execution role
role = get_execution_role()

print("Region:", region)
print("Account:", account_id)
print("Role:", role)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Region: us-east-1
Account: 275060988924
Role: arn:aws:iam::275060988924:role/service-role/AmazonSageMakerAdminIAMExecutionRole


In [22]:
from sagemaker.core import image_uris

training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2",
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="training",
)

print(training_image)

683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3


In [29]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import Compute, InputData, SourceCode

source_code = SourceCode(
    source_dir=".",
    entry_script="train_sagemaker.py",
)

compute = Compute(
    instance_type="ml.t3.medium",
    instance_count=1,
)

train_data = InputData(
    channel_name="train",
    data_source=data_s3_uri,
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    sagemaker_session=sagemaker_session,
    source_code=source_code,
    compute=compute,
    input_data_config=[train_data],
    base_job_name="wine-quality-container-training",
)

print("ModelTrainer configured successfully.")

ModelTrainer configured successfully.


In [32]:
training_job = trainer.train(wait=True)

sagemaker.config INFO - Applied value from config key = SageMaker.TrainingJob.Environment


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 training_job = trainer.train(wait=True)                                                      │
│   2                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:215 in     │
│ wrapper                                                                                          │
│                                                                                                  │
│   212 │   │   │   │   │   caught_ex = e                                                          │
│   213 │   │   │   │   finally:                                                                   │
│   214 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 215 │   │   │   │   │   │   raise caught_ex                                                    │
│   216 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   217 │   │   │   else:                                                                          │
│   218 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:181 in     │
│ wrapper                                                                                          │
│                                                                                                  │
│   178 │   │   │   │   start_timer = perf_counter()                                               │
│   179 │   │   │   │   try:                                                                       │
│   180 │   │   │   │   │   # Call the original function                                           │
│ ❱ 181 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   182 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   183 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   184 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:346 in       │
│ wrapper                                                                                          │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pydantic/_internal/

## Troubleshooting

Part A completed successfully.

For Part B, the managed SageMaker container was configured successfully using the AWS Scikit-learn training image.

When launching the SageMaker training job, AWS returned a ResourceLimitExceeded error indicating that the requested training instance was not available for the current AWS account and region.

Earlier attempts to use a custom Docker container also failed because the SageMaker execution role did not have Amazon ECR permissions (ecr:CreateRepository and ecr:DescribeRepositories).

These issues are caused by AWS account permissions and service quotas rather than errors in the notebook code. The complete error logs are included with the submission.

## Cleanup

Delete any endpoint you create and stop unused Studio applications to avoid continuing charges. The training job stops after completion.

## Evidence to capture

Include screenshots of successful notebook output, the ECR image, the SageMaker training-job status, the model-artifact S3 URI, the GitHub notebook URL, and any CloudWatch logs used for troubleshooting.